In [ ]:
from glob import glob
import pandas as pd
from functools import reduce

models  = glob('../openLLMLeaderboard/compressed/*.pkl')

In [ ]:
model_dict = {} #finds instances that are shared across all LLMs
for model in models:
    df = pd.read_pickle(model)
    df = df.rename(columns={'doc_hash': 'sample_id','input': 'prompt'})
    name = model.split('/')[-1].split('.pkl')[0]
    model_dict[name] = df

hash_sets = [set(df['sample_id']) for df in model_dict.values()]
shared_hashes = set.intersection(*hash_sets)

In [ ]:
trains = []
tests = []

for name, df in model_dict.items():
    train = df[~df['sample_id'].isin(shared_hashes)]   
    test  = df[df['sample_id'].isin(shared_hashes)]    

    train = train.drop_duplicates(subset=['sample_id'], keep='first')
    test = test.drop_duplicates(subset=['sample_id'], keep='first')

    train = train.rename(columns={'correctness': name})
    test = test.rename(columns={'correctness': name})
    
    trains.append(train)
    tests.append(test)

In [ ]:
train = reduce(lambda left, right: pd.merge(left, right, on=['sample_id', 'prompt'], how='outer'), trains)
test = reduce(lambda left, right: pd.merge(left, right, on=['sample_id', 'prompt'], how='outer'), tests)

In [ ]:
train['eval_name'] = 'openLLMLeaderboard'
test['eval_name'] = 'openLLMLeaderboard'

In [ ]:
train.to_pickle('../openLLMLeaderboard/data/train.pkl')
test.to_pickle('../openLLMLeaderboard/data/test.pkl')